### Лабораторна робота №4: Факторгрупи та гомоморфізми

**Мета роботи:** Практично ознайомитися з поняттями нормальних підгруп, факторгруп та гомоморфізмів і дослідити їх властивості за допомогою засобів SageMath.

---
### 1. Класи суміжності та нормальні підгрупи

Розглянемо групу $S_4$ та її підгрупи:<br>
$\quad$ (a) $H_1=\langle (1,2,3,4)\rangle$;<br>
$\quad$ (b) $H_2=\langle (1,3), (1,2,3,4)\rangle$;<br>
$\quad$ (c) $H_3=K_4$.

Для кожної підгрупи $H$ виконайте наступні кроки:
1. Обчисліть порядок підгрупи $H$ та її індекс у групі $[S_4 : H]$.
2. Знайдіть ліві та праві класи суміжності підгрупи $H$ у $S_4$.
3. Визначте, чи є $H$ нормальною підгрупою $S_4$.
4. Якщо $H$ є нормальною, то побудуйте факторгрупу $G/H$, її таблицю Келі та визначіть структуру.
5. Якщо $H$ не є нормальною, то знайдіть такий елемент $a\in S_4$, що $aH\neq Ha$.


In [5]:
S4 = SymmetricGroup(4)
subgroups = {
    "H1": S4.subgroup([S4("(1,2,3,4)")]),
    "H2": S4.subgroup([S4("(1,3)"), S4("(1,2,3,4)")]),
    "H3": S4.subgroup([S4("(1,2)(3,4)"), S4("(1,3)(2,4)")]),
}

for name, H in subgroups.items():
    print("=" * 70)
    print(name, "=", H.gens())
    print("Порядок:", H.order(), "| індекс:", S4.order() // H.order())

    left = S4.cosets(H, side="left")
    right = S4.cosets(H, side="right")
    print("Ліві класи суміжності:")
    for c in left:
        print("  ", [str(x) for x in c])
    print("Праві класи суміжності:")
    for c in right:
        print("  ", [str(x) for x in c])

    normal = H.is_normal(S4)
    print("Нормальна підгрупа:", normal)

    if normal:
        Q = S4.quotient(H)
        print("Факторгрупа S4/H: порядок", Q.order(), "| структура:", Q.structure_description())
        print(Q.cayley_table())
    else:
        a = next(g for g in S4 if set(g * h for h in H) != set(h * g for h in H))
        print("a =", a)
        print("aH =", sorted(str(a * h) for h in H))
        print("Ha =", sorted(str(h * a) for h in H))
        print("aH != Ha:", set(a * h for h in H) != set(h * a for h in H))

H1 = ((1,2,3,4),)
Порядок: 4 | індекс: 6
Ліві класи суміжності:
   ['()', '(1,2,3,4)', '(1,3)(2,4)', '(1,4,3,2)']
   ['(3,4)', '(1,2,3)', '(1,3,2,4)', '(1,4,2)']
   ['(2,3)', '(1,2,4)', '(1,3,4,2)', '(1,4,3)']
   ['(2,3,4)', '(1,2,4,3)', '(1,3,2)', '(1,4)']
   ['(2,4,3)', '(1,2)', '(1,3,4)', '(1,4,2,3)']
   ['(2,4)', '(1,2)(3,4)', '(1,3)', '(1,4)(2,3)']
Праві класи суміжності:
   ['()', '(1,2,3,4)', '(1,3)(2,4)', '(1,4,3,2)']
   ['(3,4)', '(1,2,4)', '(1,4,2,3)', '(1,3,2)']
   ['(2,3)', '(1,3,4)', '(1,2,4,3)', '(1,4,2)']
   ['(2,3,4)', '(1,3,2,4)', '(1,4,3)', '(1,2)']
   ['(2,4,3)', '(1,4)', '(1,2,3)', '(1,3,4,2)']
   ['(2,4)', '(1,4)(2,3)', '(1,3)', '(1,2)(3,4)']
Нормальна підгрупа: False
a = (2,3,4)
aH = ['(1,2,4,3)', '(1,3,2)', '(1,4)', '(2,3,4)']
Ha = ['(1,2)', '(1,3,2,4)', '(1,4,3)', '(2,3,4)']
aH != Ha: True
H2 = ((1,3), (1,2,3,4))
Порядок: 8 | індекс: 3
Ліві класи суміжності:
   ['()', '(2,4)', '(1,2)(3,4)', '(1,2,3,4)', '(1,3)', '(1,3)(2,4)', '(1,4,3,2)', '(1,4)(2,3)']
   ['(3,4

---
### 2. Нормальні підгрупи в діедральних групах

Розгляньте діедральні групи $D_n$ для значень $n\in\{3,4,\dots,30\}$ і виконайте такі кроки:
1. Побудуйте групу $D_n$.
2. Визначте кількість нормальних підгруп у $D_n$.
3. Складіть перелік порядків усіх нормальних підгруп.
4. Визначте алгебраїчну структуру кожної нормальної підгрупи.
5. Знайдіть закономірність і виведіть гіпотезу про кількість та структуру нормальних підгруп в $D_n$.
6. Сформулювати гіпотезу: Скільки нормальних підгруп в $D_n$ та яка їхня структура? Спрогнозувати кількість нормальних підгруп у $D_{470448}$ без виконання обчислень. 

In [6]:
from collections import Counter

rows = []
for n in range(3, 31):
    D = DihedralGroup(n)
    N = D.normal_subgroups()
    orders = sorted(x.order() for x in N)
    structs = Counter(x.structure_description() for x in N)
    rows.append((n, len(N), number_of_divisors(n), orders, dict(structs)))

for n, k, t, orders, structs in rows:
    print(f"n = {n:>2} | нормальних підгруп: {k:>2} | tau(n) = {t:>2} | порядки: {orders}")
    print(f"        структури: {structs}")

def predicted(n):
    return number_of_divisors(n) + (3 if n % 2 == 0 else 1)

print()
print("Гіпотеза: #Norm(D_n) = tau(n) + 1 для непарних n та tau(n) + 3 для парних n")
print("Збігається для всіх n = 3..30:", all(k == predicted(n) for n, k, _, _, _ in rows))
print("Структура: циклічні підгрупи <r^d> для d | n, сама D_n,")
print("а для парних n ще дві підгрупи індексу 2, ізоморфні D_(n/2)")

m = 470448
print()
print("470448 =", factor(m))
print("tau(470448) =", number_of_divisors(m))
print("Прогноз для D_470448:", predicted(m))

n =  3 | нормальних підгруп:  3 | tau(n) =  2 | порядки: [1, 3, 6]
        структури: {'S3': 1, 'C3': 1, '1': 1}
n =  4 | нормальних підгруп:  6 | tau(n) =  3 | порядки: [1, 2, 4, 4, 4, 8]
        структури: {'D4': 1, 'C4': 1, 'C2 x C2': 2, 'C2': 1, '1': 1}
n =  5 | нормальних підгруп:  3 | tau(n) =  2 | порядки: [1, 5, 10]
        структури: {'D5': 1, 'C5': 1, '1': 1}
n =  6 | нормальних підгруп:  7 | tau(n) =  4 | порядки: [1, 2, 3, 6, 6, 6, 12]
        структури: {'D6': 1, 'C6': 1, 'S3': 2, 'C2': 1, 'C3': 1, '1': 1}
n =  7 | нормальних підгруп:  3 | tau(n) =  2 | порядки: [1, 7, 14]
        структури: {'D7': 1, 'C7': 1, '1': 1}
n =  8 | нормальних підгруп:  7 | tau(n) =  4 | порядки: [1, 2, 4, 8, 8, 8, 16]
        структури: {'D8': 1, 'C8': 1, 'D4': 2, 'C4': 1, 'C2': 1, '1': 1}
n =  9 | нормальних підгруп:  4 | tau(n) =  3 | порядки: [1, 3, 9, 18]
        структури: {'D9': 1, 'C9': 1, 'C3': 1, '1': 1}
n = 10 | нормальних підгруп:  7 | tau(n) =  4 | порядки: [1, 2, 5, 10, 10, 10, 20]

---
### 3. Гомоморфізм групи матриць у симетричну групу. 

Розглянемо групу $GL_2(\mathbb{Z}_3)$ та множину прямих $M=\{\ell_1, \ell_2, \ell_3, \ell_4\}$ у просторі $\mathbb{Z}_3^2$, що визначаються парами пропорційних ненульових векторів:
\begin{align*}
\ell_1 = \{ (1,0), (2,0) \}, \
\ell_2 = \{ (0,1), (0,2) \}, \
\ell_3 = \{ (1,1), (2,2) \}, \
\ell_4 = \{ (1,2), (2,1) \}.
\end{align*}

1. Побудуйте групу $GL_2(\mathbb{Z}_3)$.
2. Задайте відображення
    $$
    \phi: GL_2(\mathbb{Z}_3) \longrightarrow S_4,
    $$
    яке переводить матрицю у перестановку, що описує її дію на прямі з $M$.
3. Перевірте, що $\phi$ є гомоморфізмом груп.<br>
4. Обчисліть ядро $N=\ker(\phi)$ та образ $\mathrm{Im}(\phi)$.
5. Побудуйте факторгрупу $GL_2(\mathbb{Z}_3)/N$ та порівняйте її з образом $\mathrm{Im}(\phi)$.



In [7]:
F = GF(3)
GL23 = GL(2, F)
S4 = SymmetricGroup(4)
elems = list(GL23)
print("|GL_2(Z_3)| =", GL23.order())

def norm(v):
    return tuple(F(x) for x in v)

lines = [
    frozenset([norm((1, 0)), norm((2, 0))]),
    frozenset([norm((0, 1)), norm((0, 2))]),
    frozenset([norm((1, 1)), norm((2, 2))]),
    frozenset([norm((1, 2)), norm((2, 1))]),
]

def act(M, L):
    return frozenset(tuple(vector(F, v) * M) for v in L)

def phi(g):
    M = g.matrix()
    imgs = [lines.index(act(M, L)) + 1 for L in lines]
    return S4(Permutation(imgs))

print("phi є гомоморфізмом:", all(phi(a * b) == phi(a) * phi(b) for a in elems for b in elems))

N = [g for g in elems if phi(g) == S4.one()]
Im = S4.subgroup([phi(g) for g in GL23.gens()])
print("Ядро N:")
for g in N:
    print(g.matrix().rows())
print("|N| =", len(N))
print("Ядро збігається з центром GL_2(Z_3):", set(N) == set(GL23.center()))
print("|Im(phi)| =", Im.order(), "| структура:", Im.structure_description())
print("Im(phi) = S4:", Im.order() == S4.order())

cosets = {frozenset(g * n for n in N) for g in elems}
induced = {c: phi(next(iter(c))) for c in cosets}
print("|GL_2(Z_3)/N| =", len(cosets))
print("Індуковане відображення клас -> Im ін'єктивне:", len(set(induced.values())) == len(cosets))
print("Образ індукованого відображення = Im:", set(induced.values()) == set(Im))
print("|GL_2(Z_3)/N| == |Im(phi)|:", len(cosets) == Im.order())
print("|N| * |Im| == |GL_2(Z_3)|:", len(N) * Im.order() == GL23.order())
print("Отже GL_2(Z_3)/N ≅ Im(phi) ≅ S4")

|GL_2(Z_3)| = 48


phi є гомоморфізмом: True
Ядро N:
[(1, 0), (0, 1)]
[(2, 0), (0, 2)]
|N| = 2
Ядро збігається з центром GL_2(Z_3): True
|Im(phi)| = 24 | структура: S4
Im(phi) = S4: True
|GL_2(Z_3)/N| = 24
Індуковане відображення клас -> Im ін'єктивне: True
Образ індукованого відображення = Im: True
|GL_2(Z_3)/N| == |Im(phi)|: True
|N| * |Im| == |GL_2(Z_3)|: True
Отже GL_2(Z_3)/N ≅ Im(phi) ≅ S4


---
### 4. Група Хігмана-Сімса

Перейдіть на сторінку ATLAS of Finite Groups (Higman--Sims group, представлення на 100 точках):<div align="center">https://brauer.maths.qmul.ac.uk/Atlas/v3/spor/HS</div>

і в розділі Representations of HS : Number of Points 100 знайдіть два твірні елементи, задані підстановками.

1. Задайте ці підстановки як елементи симетричної групи на $100$ точках.
2. Побудуйте підгрупу $H$, породжену знайденими елементами. Ця підгрупа реалізує групу Хігмана-Сімса.
3. Обчисліть порядок групи $H$.
4. Знайдіть усі нормальні підгрупи $H$.
5. Зробіть висновки щодо структури групи $H$ (зокрема, визначте, чи є вона простою). 

In [9]:
import re

a_text = """a11 := (1,60)(2,72)(3,81)(4,43)(5,11)(6,87)(7,34)(9,63)(12,46)(13,28)(14,71)
     (15,42)(16,97)(18,57)(19,52)(21,32)(23,47)(24,54)(25,83)(26,78)(29,89)
     (30,39)(33,61)(35,56)(37,67)(44,76)(45,88)(48,59)(49,86)(50,74)(51,66)
     (53,99)(55,75)(62,73)(65,79)(68,82)(77,92)(84,90)(85,98)(94,100);"""
b_text = """a21 := (1,86,13,10,47)(2,53,30,8,38)(3,40,48,25,17)(4,29,92,88,43)(5,98,66,54,
     65)(6,27,51,73,24)(7,83,16,20,28)(9,23,89,95,61)(11,42,46,91,32)(12,14,
     81,55,68)(15,90,31,56,37)(18,69,45,84,76)(19,59,79,35,93)(21,22,64,39,
     100)(26,58,96,85,77)(33,52,94,75,44)(34,62,87,78,50)(36,82,60,74,72)(41,
     80,70,49,67)(57,63,71,99,97);"""

assert a_text.strip() and b_text.strip(), "Вставте у a_text і b_text підстановки a та b зі сторінки ATLAS (HSG1-p100B0)"

def parse(t):
    t = t.strip()
    t = re.sub(r"^[^(]*", "", t)
    t = t.replace("\n", "").replace(" ", "").replace("\\", "").rstrip(";")
    return t

S100 = SymmetricGroup(100)
a = S100(parse(a_text))
b = S100(parse(b_text))
print("порядок a:", a.order(), "| порядок b:", b.order(), "| порядок ab:", (a * b).order())

H = S100.subgroup([a, b])
print("|H| =", H.order())
print("Факторизація |H|:", factor(H.order()))
print("|HS| = 44352000:", H.order() == 44352000)

NS = H.normal_subgroups()
print("Кількість нормальних підгруп:", len(NS))
for K in NS:
    print("  порядок", K.order())
print("H проста:", H.is_simple())
print("Центр H тривіальний:", H.center().order() == 1)

порядок a: 2 | порядок b: 5 | порядок ab: 11
|H| = 44352000
Факторизація |H|: 2^9 * 3^2 * 5^3 * 7 * 11
|HS| = 44352000: True
Кількість нормальних підгруп: 2
  порядок 1
  порядок 44352000
H проста: True
Центр H тривіальний: True


---
### 5*. Проекції групи $SL_2(\mathbb{Z})$ на $SL_2(\mathbb{Z}_n)$ 

Розглянемо спеціальну лінійну групу $SL_2(\mathbb{Z})$, яка породжується двома матрицями:
$$S = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix},\qquad T = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}. $$

1. Задайте гомоморфізм $\rho_n: SL_2(\mathbb{Z}) \to SL_2(\mathbb{Z}_n)$, який редукує матриці за модулем $n$. 
2. Для значень $n\in\{2, 3, \ldots, 10\}$ перевірте, чи є $\rho_n$ сюр'єктивним, виконавши такі кроки:<br>
    a. Побудуйте групу $G_n = SL_2(\mathbb{Z}_n)$.<br>
    b. Побудуйте групу $H_n = \langle \rho_n(S), \rho_n(T) \rangle$.<br>
    c. Порівняйте порядки груп $H_n$ та $G_n$.



---
### 6*. Автоморфізми симетричних груп $S_n$

Для майже всіх симетричних груп $S_n$ усі автоморфізми є внутрішніми, тобто $\text{Aut}(S_n) = \text{Inn}(S_n)$. Однак існує одне особливе значення $n$, для якого з'являється зовнішній автоморфізм.

1. Для кожного $n=2,3,4,\ldots,8$:<br>
   (а) Побудуйте групу $S_n$ у SageMath (через GAP).<br>
   (б) Знайдіть групу автоморфізмів $Aut(S_n)$ та внутрішніх автоморфізмів $Inn(S_n)$.<br>
   (в) Порівняйте їх порядки та визначте, для якого $n$ вони відрізняються.<br>
2. Для знайденого значення $n$:<br>
   (a) Побудуйте невнутрішній автоморфізм групи $S_n$.<br>
   (б) Покажіть, як він діє на твірних групи (наприклад, на сусідніх транспозиціях $(1\,2), (2\,3),\ldots, (n-1\,n)$.
